# 02 - Clean PC Builder Component Candidates

Use this notebook after `01_explore_products_cleaned.ipynb`.

It filters `products_cleaned.csv` into PC Builder component candidates and exports reviewable files:

- `data/component_candidates.csv`
- `data/component_candidates.json`
- `data/cleaning_reports/component_cleaning_report.json`
- `data/cleaning_reports/component_rejects.csv`
- `data/cleaning_reports/component_spec_gaps.csv`

By default it does **not** overwrite `data/components.json`.

## Cleaning philosophy

The CSV is raw marketplace data, so this notebook keeps the pipeline conservative:

- Keep only PC Builder-relevant source categories.
- Normalize raw marketplace categories into app categories.
- Drop obvious accessories, external-only storage, NAS systems, monitor adapters, and TV rows.
- Preserve SKU, EnterKomputer URL, future marketplace URLs, image URL, and scrape timestamp.
- Extract lightweight compatibility specs with deterministic regexes.
- Export candidates for review before replacing runtime data.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import display

CSV_PATH = Path('products_cleaned.csv')
DATA_DIR = Path('data')
REPORT_DIR = DATA_DIR / 'cleaning_reports'
CANDIDATE_CSV = DATA_DIR / 'component_candidates.csv'
CANDIDATE_JSON = DATA_DIR / 'component_candidates.json'
RUNTIME_COMPONENTS_JSON = DATA_DIR / 'components.json'
CURATED_RAM_JSON = DATA_DIR / 'curated_ram.json'

DATA_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_RUNTIME_COMPONENTS = False  # Change to True only after reviewing candidate exports and validation results.
MERGE_CURATED_RAM = True

if not CSV_PATH.exists():
    try:
        from google.colab import files
        print('Upload products_cleaned.csv')
        uploaded = files.upload()
        if uploaded:
            CSV_PATH = Path(next(iter(uploaded.keys())))
    except Exception as exc:
        raise FileNotFoundError('products_cleaned.csv was not found. Upload it or place it beside the notebook.') from exc

print('Using:', CSV_PATH)

In [ ]:
raw = pd.read_csv(CSV_PATH)
raw.columns = [c.strip() for c in raw.columns]

EXPECTED_COLUMNS = [
    'sku', 'name', 'category', 'subcategory', 'price_idr', 'stock_status',
    'description', 'specifications', 'image_url', 'product_url',
    'tokopedia_url', 'shopee_url', 'scraped_at',
]

for col in EXPECTED_COLUMNS:
    if col not in raw.columns:
        raw[col] = ''

raw['sku'] = raw['sku'].astype(str).str.strip()
raw['name'] = raw['name'].fillna('').astype(str).str.strip()
raw['category'] = raw['category'].fillna('').astype(str).str.strip()
raw['subcategory'] = raw['subcategory'].fillna('').astype(str).str.strip()
raw['stock_status'] = raw['stock_status'].fillna('').astype(str).str.strip()
raw['description'] = raw['description'].fillna('').astype(str).str.strip()
raw['specifications'] = raw['specifications'].fillna('').astype(str).str.strip()
raw['price_idr'] = pd.to_numeric(raw.get('price_idr'), errors='coerce').fillna(0).astype('int64')
for url_col in ['image_url', 'product_url', 'tokopedia_url', 'shopee_url']:
    raw[url_col] = raw[url_col].fillna('').astype(str).str.strip()
raw['scraped_at'] = raw['scraped_at'].fillna('').astype(str).str.strip()

print('Raw rows:', len(raw))
print('Columns:', ', '.join(raw.columns))
display(raw.head(5))

In [ ]:
CATEGORY_MAP = {
    'Processor': 'cpu',
    'CPU': 'cpu',
    'Motherboard': 'motherboard',
    'Mainboard': 'motherboard',
    'RAM': 'ram',
    'Memory': 'ram',
    'VGA': 'gpu',
    'GPU': 'gpu',
    'Graphic Card': 'gpu',
    'Graphics Card': 'gpu',
    'SSD': 'ssd',
    'Hard Drive': 'hdd',
    'HDD': 'hdd',
    'PSU': 'psu',
    'Power Supply': 'psu',
    'Cooler': 'cooler',
    'Cooling': 'cooler',
    'Fan': 'cooler',
    'Casing': 'case',
    'Case': 'case',
    'LCD': 'monitor',
    'Monitor': 'monitor',
    'UPS': 'ups',
}

REAL_VENDOR_BRANDS = [
    'ASUS', 'ASRock', 'MSI', 'Gigabyte', 'Biostar', 'Colorful', 'Palit', 'Galax', 'Zotac',
    'PNY', 'Inno3D', 'Sapphire', 'PowerColor', 'XFX', 'Corsair', 'Cooler Master', 'NZXT',
    'Fractal', 'Lian Li', 'Phanteks', 'be quiet!', 'Thermaltake', 'Seasonic', 'FSP', 'Antec',
    'Deepcool', 'ID-COOLING', 'Noctua', 'Arctic', 'Samsung', 'WD', 'Western Digital',
    'Seagate', 'Crucial', 'Kingston', 'Adata', 'Team', 'TeamGroup', 'Klevv', 'Patriot',
    'Lexar', 'SilverStone', 'Aerocool', 'Cougar', 'InWin', 'Montech', 'Tecware', 'LG',
    'AOC', 'ViewSonic', 'Dell', 'BenQ', 'Acer', 'Lenovo', 'APC', 'ICA', 'Eaton',
]
PLATFORM_BRANDS = ['AMD', 'Intel', 'Nvidia']
KNOWN_BRANDS = REAL_VENDOR_BRANDS + PLATFORM_BRANDS

EXTERNAL_STORAGE_RE = re.compile(r'eksternal|external|without adapter|must use adapter|enclosure|docking|bracket|case hdd|adapter', re.I)
NAS_OR_HDD_ACCESSORY_RE = re.compile(r'\bnas\b|qnap|synology|asustor|terra ?master|rackmount|rail|bay|enclosure|docking|external|bracket|surveillance system|nvr', re.I)
COOLER_ACCESSORY_RE = re.compile(r'mounting|accessor|bracket|thermal paste|vga cooler|ssd cooler|memory cooler|controller only|hub only', re.I)
PSU_ACCESSORY_RE = re.compile(r'cable|adapter|connector|extension|sleeved|\d+\s*-?\s*pin\s+to', re.I)
MONITOR_ACCESSORY_RE = re.compile(r'adapter|adaptor|bracket|mount|wall mount|arm|stand|remote|cable|converter|panel only|sparepart|power supply', re.I)
TV_RE = re.compile(r'\btv\b|smart tv|android tv|oled tv|qled tv', re.I)


def normalize_app_category(source_category):
    return CATEGORY_MAP.get(str(source_category or '').strip())


def detect_brand(name):
    text = str(name or '').strip()
    upper = f' {text.upper()} '
    for brand in sorted(REAL_VENDOR_BRANDS, key=len, reverse=True):
        b = brand.upper()
        if text.upper().startswith(b) or f' {b} ' in upper:
            return brand
    for brand in PLATFORM_BRANDS:
        b = brand.upper()
        if text.upper().startswith(b) or f' {b} ' in upper:
            return brand
    return None


def clean_url(value):
    value = str(value or '').strip()
    if not value or value.lower() in {'nan', 'none', 'null'}:
        return None
    return value


def clean_image(value):
    value = clean_url(value)
    if not value or 'noimage' in value.lower():
        return None
    return value


def build_marketplace_links(row):
    links = []
    for marketplace, column in [
        ('enterkomputer', 'product_url'),
        ('tokopedia', 'tokopedia_url'),
        ('shopee', 'shopee_url'),
    ]:
        url = clean_url(row.get(column, ''))
        if url:
            links.append({'marketplace': marketplace, 'url': url})
    return links


def parse_spec_json(value):
    if isinstance(value, dict):
        return value
    text = str(value or '').strip()
    if not text or text in {'{}', 'nan', 'None'}:
        return {}
    try:
        parsed = json.loads(text)
        return parsed if isinstance(parsed, dict) else {}
    except json.JSONDecodeError:
        return {}

In [ ]:
def parse_cpu(name, subcategory, source_specs=None):
    text = f'{name} {subcategory}'
    lower = text.lower()
    socket = None
    for pattern in [r'LGA\s?1851', r'LGA\s?1700', r'LGA\s?1200', r'LGA\s?1151', r'LGA\s?1150', r'AM5', r'AM4', r'TRX40', r'sTRX4']:
        m = re.search(pattern, text, re.I)
        if m:
            socket = m.group(0).replace(' ', '').upper().replace('LGA', 'LGA ')
            break
    brand = 'AMD' if 'ryzen' in lower or re.search(r'\bamd\b', lower) else 'Intel' if 'intel' in lower or 'core i' in lower or 'core ultra' in lower else None
    cores = None
    core_match = re.search(r'(\d+)\s*(?:core|cores)', text, re.I)
    if core_match:
        cores = int(core_match.group(1))
    else:
        for fam, count in [('ryzen 9', 12), ('ryzen 7', 8), ('ryzen 5', 6), ('ryzen 3', 4), ('core ultra 9', 24), ('core ultra 7', 20), ('core ultra 5', 14), ('core i9', 16), ('core i7', 8), ('core i5', 6), ('core i3', 4)]:
            if fam in lower:
                cores = count
                break
    return {'socket': socket, 'brand': brand, 'cores': cores}


def parse_motherboard(name, subcategory, source_specs=None):
    text = f'{name} {subcategory}'
    lower = text.lower()
    socket = None
    for pattern in [r'LGA\s?1851', r'LGA\s?1700', r'LGA\s?1200', r'LGA\s?1151', r'LGA\s?1150', r'AM5', r'AM4', r'FM2\+?', r'TRX40']:
        m = re.search(pattern, text, re.I)
        if m:
            socket = m.group(0).replace(' ', '').upper().replace('LGA', 'LGA ')
            break
    form_factor = 'ATX'
    if re.search(r'mini[- ]?itx|\bitx\b', lower):
        form_factor = 'ITX'
    elif re.search(r'micro[- ]?atx|m[- ]?atx|\bmatx\b', lower):
        form_factor = 'mATX'
    elif re.search(r'e[- ]?atx|\beatx\b', lower):
        form_factor = 'EATX'
    ram_type = 'DDR5' if 'ddr5' in lower else 'DDR4' if 'ddr4' in lower else 'DDR3' if 'ddr3' in lower else None
    return {'socket': socket, 'form_factor': form_factor, 'ram_type': ram_type}


def parse_ram(name, subcategory, source_specs=None):
    text = f'{name} {subcategory}'
    lower = text.lower()
    ram_type = 'DDR5' if 'ddr5' in lower else 'DDR4' if 'ddr4' in lower else 'DDR3' if 'ddr3' in lower else None
    capacity_gb = None
    modules = None
    kit = re.search(r'(\d+)\s*[xX]\s*(\d+)\s*GB', text)
    if kit:
        modules = int(kit.group(1))
        capacity_gb = modules * int(kit.group(2))
    else:
        m = re.search(r'(\d+)\s*GB', text, re.I)
        if m:
            capacity_gb = int(m.group(1))
            modules = 1
    speed_mhz = None
    m = re.search(r'(?:DDR[345][- ]?)?(\d{4,5})\s*(?:MHz)?', text, re.I)
    if m:
        value = int(m.group(1))
        if value >= 1000:
            speed_mhz = value
    return {'type': ram_type, 'capacity_gb': capacity_gb, 'speed_mhz': speed_mhz, 'modules': modules}


def parse_gpu(name, subcategory, source_specs=None):
    text = f'{name} {subcategory}'
    lower = f' {text.lower()} '
    vendor = 'Nvidia' if any(k in lower for k in ['nvidia', 'geforce', ' rtx', ' gtx']) else 'AMD' if any(k in lower for k in ['radeon', ' rx ']) else 'Intel' if ' arc ' in lower else None
    vram = None
    m = re.search(r'(\d+)\s*GB', text, re.I)
    if m:
        vram = int(m.group(1))
    rec_psu = None
    m = re.search(r'(?:RTX|GTX|RX)\s*([0-9]{3,4})', text, re.I)
    if m:
        num = int(m.group(1))
        rec_psu = 850 if num >= 4080 or num >= 7900 else 750 if num >= 4070 or num >= 7800 else 650 if num >= 4060 or num >= 7600 else 550
    return {'vendor': vendor, 'vram_gb': vram, 'recommended_psu_w': rec_psu}


def parse_storage(name, subcategory, category, source_specs=None):
    text = f'{name} {subcategory}'
    lower = text.lower()
    capacity_gb = None
    m = re.search(r'(\d+(?:\.\d+)?)\s*TB', text, re.I)
    if m:
        capacity_gb = int(float(m.group(1)) * 1024)
    else:
        m = re.search(r'(\d+)\s*GB', text, re.I)
        if m:
            capacity_gb = int(m.group(1))
    interface = 'NVMe' if any(k in lower for k in ['nvme', 'pcie', 'm.2']) else 'SATA' if 'sata' in lower else None
    rpm = None
    m = re.search(r'(5400|7200|10000)\s*RPM', text, re.I)
    if m:
        rpm = int(m.group(1))
    return {'capacity_gb': capacity_gb, 'interface': interface, 'rpm': rpm} if category == 'hdd' else {'capacity_gb': capacity_gb, 'interface': interface}


def parse_psu(name, subcategory, source_specs=None):
    text = f'{name} {subcategory}'
    wattage = None
    m = re.search(r'(\d{3,4})\s*(?:W|Watt)', text, re.I)
    if m:
        wattage = int(m.group(1))
    rating = None
    for value in ['Titanium', 'Platinum', 'Gold', 'Silver', 'Bronze', 'White']:
        if value.lower() in text.lower():
            rating = value
            break
    modular = 'full' if re.search(r'full modular', text, re.I) else 'semi' if re.search(r'semi modular', text, re.I) else None
    return {'wattage_w': wattage, 'rating': rating, 'modular': modular}


def parse_cooler(name, subcategory, source_specs=None):
    text = f'{name} {subcategory}'
    lower = text.lower()
    kind = 'fan' if 'fan casing' in lower or re.search(r'\bfan\b', lower) else 'liquid' if any(k in lower for k in ['aio', 'liquid', 'water']) else 'air'
    fan_size = None
    m = re.search(r'(\d{2,3})\s*mm|\b(8|12|14|18)cm\b', text, re.I)
    if m:
        fan_size = int(m.group(1) or int(m.group(2)) * 10)
    return {'type': kind, 'fan_size_mm': fan_size}


def parse_case(name, subcategory, source_specs=None):
    text = f'{name} {subcategory}'.lower()
    max_form = 'EATX' if 'e-atx' in text or 'eatx' in text else 'ITX' if 'itx' in text else 'mATX' if 'matx' in text or 'micro' in text else 'ATX'
    return {'max_form_factor': max_form, 'form_factor': max_form}


def parse_monitor(name, subcategory, source_specs=None):
    text = f'{name} {subcategory}'
    size = None
    m = re.search(r'(\d{2}(?:\.\d)?)\s*(?:inch|")', text, re.I)
    if m:
        size = float(m.group(1))
    refresh = None
    m = re.search(r'(\d{2,3})\s*Hz', text, re.I)
    if m:
        refresh = int(m.group(1))
    resolution = None
    for value in ['4K', 'QHD', 'WQHD', '1440p', '1080p', 'FHD', 'UHD']:
        if value.lower() in text.lower():
            resolution = value
            break
    return {'size_inches': size, 'refresh_hz': refresh, 'resolution': resolution}


def parse_ups(name, subcategory, source_specs=None):
    text = f'{name} {subcategory}'
    va = None
    m = re.search(r'(\d{3,5})\s*VA|([0-9]+(?:\.[0-9]+)?)\s*KVA', text, re.I)
    if m:
        va = int(m.group(1)) if m.group(1) else int(float(m.group(2)) * 1000)
    return {'capacity_va': va}


def parse_specs(category, name, subcategory, source_specs=None):
    if category == 'cpu': return parse_cpu(name, subcategory, source_specs)
    if category == 'motherboard': return parse_motherboard(name, subcategory, source_specs)
    if category == 'ram': return parse_ram(name, subcategory, source_specs)
    if category == 'gpu': return parse_gpu(name, subcategory, source_specs)
    if category in {'ssd', 'hdd'}: return parse_storage(name, subcategory, category, source_specs)
    if category == 'psu': return parse_psu(name, subcategory, source_specs)
    if category == 'cooler': return parse_cooler(name, subcategory, source_specs)
    if category == 'case': return parse_case(name, subcategory, source_specs)
    if category == 'monitor': return parse_monitor(name, subcategory, source_specs)
    if category == 'ups': return parse_ups(name, subcategory, source_specs)
    return {}

In [ ]:
def include_row(row):
    source_category = row['category']
    source_subcategory = row['subcategory']
    name = row['name']
    category = normalize_app_category(source_category)
    full_text = f'{name} {source_subcategory}'

    if not category:
        return False, None, 'source category not in PC Builder scope'
    if not name:
        return False, category, 'missing name'
    if int(row['price_idr']) <= 0:
        return False, category, 'missing price'
    if category in {'ssd', 'hdd'} and EXTERNAL_STORAGE_RE.search(full_text):
        return False, category, 'external/accessory storage'
    if category == 'hdd' and NAS_OR_HDD_ACCESSORY_RE.search(full_text):
        return False, category, 'nas or hdd accessory'
    if category == 'cooler' and COOLER_ACCESSORY_RE.search(full_text):
        return False, category, 'cooler accessory'
    if category == 'psu' and PSU_ACCESSORY_RE.search(name) and not re.search(r'\d{3,4}\s*(?:W|Watt)', name, re.I):
        return False, category, 'psu accessory'
    if category == 'monitor' and (MONITOR_ACCESSORY_RE.search(full_text) or TV_RE.search(full_text)):
        return False, category, 'monitor accessory or tv'
    return True, category, 'included'


def component_record_from_row(row, normalized_category):
    source_specs = parse_spec_json(row.get('specifications', ''))
    specs = parse_specs(normalized_category, row['name'], row['subcategory'], source_specs)
    image_url = clean_image(row.get('image_url'))
    product_url = clean_url(row.get('product_url'))
    tokopedia_url = clean_url(row.get('tokopedia_url'))
    shopee_url = clean_url(row.get('shopee_url'))
    return {
        'sku': row['sku'],
        'name': row['name'],
        'brand': detect_brand(row['name']),
        'category': normalized_category,
        'source_category': row['category'],
        'subcategory': row['subcategory'] or None,
        'price_idr': int(row['price_idr']),
        'stock_status': row['stock_status'] or None,
        'description': row.get('description') or None,
        'image_url': image_url,
        'image_path': image_url,
        'product_url': product_url,
        'tokopedia_url': tokopedia_url,
        'shopee_url': shopee_url,
        'marketplace_links': build_marketplace_links(row),
        'source': 'enterkomputer',
        'scraped_at': row.get('scraped_at') or None,
        'source_specs': source_specs,
        'specs': {k: v for k, v in specs.items() if v is not None},
    }

records = []
rejects = []

for _, row in raw.iterrows():
    keep, normalized_category, reason = include_row(row)
    if not keep:
        rejects.append({
            'sku': row['sku'],
            'name': row['name'],
            'source_category': row['category'],
            'source_subcategory': row['subcategory'],
            'normalized_category': normalized_category,
            'reason': reason,
        })
        continue
    records.append(component_record_from_row(row, normalized_category))

if MERGE_CURATED_RAM and CURATED_RAM_JSON.exists():
    curated_ram = json.loads(CURATED_RAM_JSON.read_text(encoding='utf-8'))
    existing_skus = {item['sku'] for item in records}
    added_ram = 0
    for item in curated_ram:
        sku = str(item.get('sku', '')).strip()
        if sku and sku not in existing_skus:
            merged = dict(item)
            merged.setdefault('category', 'ram')
            merged.setdefault('source', 'curated_ram')
            merged.setdefault('marketplace_links', build_marketplace_links(merged))
            merged.setdefault('image_url', merged.get('image_path'))
            records.append(merged)
            added_ram += 1
    print('Merged curated RAM rows:', added_ram)

components = pd.DataFrame(records)
rejects_df = pd.DataFrame(rejects)

print('Included component candidates:', len(components))
print('Rejected rows:', len(rejects_df))
if len(components):
    display(components.groupby('category').size().reset_index(name='rows').sort_values('rows', ascending=False))
    display(components.head(20))
else:
    display(components)

In [ ]:
# Quality checks before export.
issues = {}
issues['duplicate_sku_rows'] = int(components.duplicated('sku').sum()) if len(components) else 0
issues['missing_brand_rows'] = int(components['brand'].isna().sum()) if len(components) and 'brand' in components else 0
issues['missing_product_url_rows'] = int(components['product_url'].isna().sum()) if len(components) and 'product_url' in components else 0
issues['missing_marketplace_link_rows'] = int(components['marketplace_links'].apply(lambda links: len(links) == 0).sum()) if len(components) and 'marketplace_links' in components else 0
issues['zero_price_rows'] = int((components['price_idr'] <= 0).sum()) if len(components) else 0

required_spec_by_category = {
    'cpu': ['socket'],
    'motherboard': ['socket', 'ram_type'],
    'ram': ['type', 'capacity_gb'],
    'gpu': ['vendor'],
    'ssd': ['capacity_gb'],
    'hdd': ['capacity_gb'],
    'psu': ['wattage_w'],
    'cooler': ['type'],
    'case': ['max_form_factor'],
    'monitor': [],
    'ups': [],
}

spec_gaps = []
for item in records:
    for key in required_spec_by_category.get(item.get('category'), []):
        if key not in item.get('specs', {}):
            spec_gaps.append({'sku': item.get('sku'), 'name': item.get('name'), 'category': item.get('category'), 'missing_spec': key})

spec_gaps_df = pd.DataFrame(spec_gaps)
print('Issues:', issues)
print('Spec gap rows:', len(spec_gaps_df))
display(spec_gaps_df.head(50))

In [ ]:
# Export candidates for review.
export_columns = [
    'sku', 'name', 'brand', 'category', 'source_category', 'subcategory', 'price_idr',
    'stock_status', 'image_url', 'image_path', 'product_url', 'tokopedia_url', 'shopee_url',
    'marketplace_links', 'source', 'scraped_at', 'specs'
]

components_for_csv = components.copy()
for col in export_columns:
    if col not in components_for_csv.columns:
        components_for_csv[col] = None
components_for_csv['marketplace_links'] = components_for_csv['marketplace_links'].apply(lambda x: json.dumps(x, ensure_ascii=False))
components_for_csv['specs'] = components_for_csv['specs'].apply(lambda x: json.dumps(x, ensure_ascii=False))
components_for_csv[export_columns].to_csv(CANDIDATE_CSV, index=False)

CANDIDATE_JSON.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding='utf-8')
rejects_df.to_csv(REPORT_DIR / 'component_rejects.csv', index=False)
spec_gaps_df.to_csv(REPORT_DIR / 'component_spec_gaps.csv', index=False)

report = {
    'source_csv': str(CSV_PATH),
    'raw_rows': int(len(raw)),
    'included_rows': int(len(records)),
    'rejected_rows': int(len(rejects_df)),
    'category_counts': components.groupby('category').size().sort_values(ascending=False).to_dict() if len(components) else {},
    'issues': issues,
    'spec_gap_rows': int(len(spec_gaps_df)),
    'outputs': {
        'candidate_csv': str(CANDIDATE_CSV),
        'candidate_json': str(CANDIDATE_JSON),
        'rejects_csv': str(REPORT_DIR / 'component_rejects.csv'),
        'spec_gaps_csv': str(REPORT_DIR / 'component_spec_gaps.csv'),
    },
}
(REPORT_DIR / 'component_cleaning_report.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')

print('Wrote:', CANDIDATE_CSV)
print('Wrote:', CANDIDATE_JSON)
print('Wrote reports to:', REPORT_DIR)

In [ ]:
# Optional runtime export. Keep disabled until you have reviewed component_candidates.json and validation results.
if WRITE_RUNTIME_COMPONENTS:
    RUNTIME_COMPONENTS_JSON.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Overwrote runtime component catalog:', RUNTIME_COMPONENTS_JSON)
else:
    print('Runtime component catalog was not overwritten. Review candidates first, then set WRITE_RUNTIME_COMPONENTS = True if desired.')

## Next review steps

After running the notebook:

1. Open `data/cleaning_reports/component_cleaning_report.json`.
2. Review `data/cleaning_reports/component_spec_gaps.csv`.
3. Spot-check `data/component_candidates.csv` by category.
4. Run `03_validate_component_candidates.ipynb`.
5. Only then decide whether to replace `data/components.json`.